In [ ]:
"""
05_error_analysis.py -- ScoutAI Error Analysis (Dual-Model Edition)

Analyzes the prediction errors for both models (Full and Performance-Only)
to identify the most confusing players for the AI. Automatically routes
players to the correct model to ensure errors are calculated fairly based
on production inference logic.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

MODELS_DIR = "models"
DATA_DIR = "data"

# Ensure output directory exists
os.makedirs(DATA_DIR, exist_ok=True)

def main():
    # ==========================================
    # 1. DATABASE CONNECTION & DATA LOAD
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Connecting to database...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    # Only analyze players with a real market value.
    df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
    df = df[df['current_market_value'] > 0].copy()

    # ==========================================
    # 2. ROBUST FEATURE ENGINEERING
    # ==========================================
    print("[SYSTEM] Engineering features...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    FULL_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
        'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
        'detailed_position', 'foot', 'passport_tier',
    ]

    PERFORMANCE_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
    ]

    # ==========================================
    # 3. MODEL LOADING & PREDICTIONS
    # ==========================================
    print("[SYSTEM] Loading models and generating predictions...")
    try:
        model_full = joblib.load(os.path.join(MODELS_DIR, 'scout_model_full.pkl'))
        model_perf = joblib.load(os.path.join(MODELS_DIR, 'scout_model_performance_only.pkl'))
    except FileNotFoundError as e:
        print(f"[ERROR] Could not load models from '{MODELS_DIR}'. Please train models first.")
        sys.exit(1)

    df['predicted_market_value'] = np.nan
    df['model_used'] = ""

    has_history_mask = df['has_transfer_history'] == 1
    no_history_mask = ~has_history_mask

    if has_history_mask.any():
        log_preds_full = model_full.predict(df.loc[has_history_mask, FULL_FEATURES])
        df.loc[has_history_mask, 'predicted_market_value'] = np.expm1(log_preds_full)
        df.loc[has_history_mask, 'model_used'] = 'full'

    if no_history_mask.any():
        log_preds_perf = model_perf.predict(df.loc[no_history_mask, PERFORMANCE_FEATURES])
        df.loc[no_history_mask, 'predicted_market_value'] = np.expm1(log_preds_perf)
        df.loc[no_history_mask, 'model_used'] = 'performance_only'

    # Error Calculation: Difference between actual value and model's prediction
    df['prediction_error'] = abs(df['current_market_value'] - df['predicted_market_value'])

    # ==========================================
    # 4. REPORT GENERATION & EXPORT
    # ==========================================
    print("[SYSTEM] Generating error analysis report...")
    pd.options.display.float_format = '{:,.0f}'.format

    columns_to_show = ['player_name', 'age', 'model_used', 'current_market_value',
                       'predicted_market_value', 'prediction_error']
    rename_map = {
        'player_name': 'Player Name',
        'age': 'Age',
        'model_used': 'Model',
        'current_market_value': 'Actual Value (€)',
        'predicted_market_value': 'ScoutAI Value (€)',
        'prediction_error': 'Gap (Error €)',
    }

    report_lines = []
    report_lines.append("ScoutAI: Error Analysis Report\n")
    report_lines.append("=========================================\n")

    for label in ['full', 'performance_only']:
        subset = df[df['model_used'] == label].sort_values(by='prediction_error', ascending=False)
        presentation_df = subset[columns_to_show].rename(columns=rename_map)
        
        section_title = f"\nTop 10 Most Confusing Players -- {label} model:\n" + "-" * 95
        print(section_title)
        report_lines.append(section_title + "\n")
        
        table_str = presentation_df.head(10).to_string(index=False)
        print(table_str)
        report_lines.append(table_str + "\n")

    # Combined view
    combined_title = "\nTop 10 Most Confusing Players -- combined (both models):\n" + "-" * 95
    print(combined_title)
    report_lines.append(combined_title + "\n")
    
    combined = df.sort_values(by='prediction_error', ascending=False)
    combined_table_str = combined[columns_to_show].rename(columns=rename_map).head(10).to_string(index=False)
    print(combined_table_str)
    report_lines.append(combined_table_str + "\n")

    # Save to TXT in DATA_DIR
    txt_export_path = os.path.join(DATA_DIR, 'error_analysis_report.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)

    print(f"\n[SUCCESS] Error analysis report saved to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Connecting to database...
[SYSTEM] Engineering features...
[SYSTEM] Loading models and generating predictions...
[SYSTEM] Generating error analysis report...

Top 10 Most Confusing Players -- full model:
-----------------------------------------------------------------------------------------------
     Player Name  Age Model  Actual Value (€)  ScoutAI Value (€)  Gap (Error €)
           Pedri   23  full       150,000,000         45,137,120    104,862,880
            Gavi   21  full        30,000,000        119,653,088     89,653,088
Matthijs de Ligt   26  full        30,000,000         87,323,112     57,323,112
         Estêvão   19  full        80,000,000         35,477,488     44,522,512
   Junior Kroupi   20  full        70,000,000         30,694,072     39,305,928
  Julián Alvarez   26  full       100,000,000         62,851,268     37,148,732
   Carlos Baleba   22  full        55,000,000         18,652,440     36,347,560
     Igor Thiago   25  full        65,000,000      